# TCC - Analise Preditiva de Risco Academico Escolar
## CRISP-DM: 3. Data Preparation

## O que acontece nesta fase

A preparacao transforma arquivos separados em uma base unica de aprendizado.

Neste notebook, o foco esta nas quatro bases que entram diretamente na modelagem atual: `notas`, `faltas`, `pagamentos` e `professores`. As bases `aluno` e `responsaveis` continuam importantes para contexto e relatorios, mas nao fazem parte do nucleo desta etapa de preparacao supervisionada.

Fluxo principal:

1. Carrega notas, faltas, pagamentos e professores.
2. Normaliza tipos, datas, notas e ordem das etapas.
3. Agrega faltas por aluno, ano, disciplina e etapa.
4. Agrega pagamentos por aluno e ano letivo.
5. Consolida professores por turma e disciplina.
6. Une tudo na base de notas.
7. Cria atributos historicos, como ultimas notas, medias e tendencia.
8. Cria os alvos `target_nota_proxima` e `target_risco_proxima`.
9. Aplica o minimo de historico por disciplina.
10. Define as colunas que entram nos modelos.

## Carregar bibliotecas e localizar o projeto

In [ ]:
from pathlib import Path
import hashlib
import sys

import pandas as pd
from IPython.display import display

In [ ]:
def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "school_predictor").exists() and (candidate / "artifacts").exists():
            return candidate.resolve()
    raise RuntimeError("Nao foi possivel localizar a raiz do projeto.")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT

## Importar as mesmas funcoes da pipeline real

Para que a demonstracao fique fiel ao sistema, este notebook reutiliza as funcoes do pacote `school_predictor`. Assim, se a regra de preparacao mudar no codigo, o notebook tambem deve ser revisado.

In [ ]:
from school_predictor.pipeline.config import PipelinePaths
from school_predictor.pipeline.data import (
    load_source_tables,
    prepare_absences,
    prepare_grades,
    prepare_payments,
    prepare_teachers,
)
from school_predictor.pipeline.dataset import build_prediction_dataset, select_model_columns
from school_predictor.pipeline import dataset as dataset_steps

## Carregar os CSVs de origem

Nesta etapa, os arquivos ja devem existir em `artifacts/database/csv/`. Eles sao a fronteira publica do projeto; as consultas SQL reais ficam fora do repositorio. A funcao `load_source_tables` carrega aqui apenas as bases usadas diretamente pela modelagem.

### Exemplo fixo do que entrou na preparacao

A fase de preparacao usa diretamente estas bases centrais da pipeline:

| base_usada_na_preparacao | arquivo | linhas | colunas |
| --- | --- | --- | --- |
| notas | media_nota_aluno.csv | 239678 | 20 |
| faltas | faltas_aluno.csv | 168051 | 22 |
| pagamentos | pagamento_aluno.csv | 220090 | 26 |
| professores | professor_disciplina.csv | 1624 | 15 |

In [ ]:
paths = PipelinePaths.from_project_root(PROJECT_ROOT, min_history=2, mode="demonstracao_data_preparation")
source_tables = load_source_tables(paths)

raw_summary = pd.DataFrame([
    {"base": name, "linhas_originais": len(frame), "colunas_originais": len(frame.columns)}
    for name, frame in source_tables.items()
])
raw_summary

### O que entrou nesta fase

A tabela acima mostra o ponto de partida real da preparação. Neste momento, os dados ainda estao separados por origem e cada base conserva sua propria granularidade.

O que vai mudar ao longo deste notebook e justamente isso: no fim da fase, o projeto deixara de trabalhar com tabelas isoladas e passara a trabalhar com uma observacao temporal unica por aluno, disciplina e etapa.

## Funcoes de visualizacao segura

Amostras exibidas neste notebook mascaram identificadores, nomes, datas pessoais e campos cadastrais. O objetivo e mostrar a forma dos dados, nao revelar pessoas.

In [ ]:
SENSITIVE_TOKENS = (
    "id", "codigo", "nomealuno", "nomeresponsavel", "nomefuncionario",
    "datanascimento", "logradouro", "quadra", "lote", "numeroresidencia",
    "complemento", "bairro", "cep",
)


def stable_mask(value, prefix: str) -> str:
    if pd.isna(value):
        return "<ausente>"
    digest = hashlib.sha256(str(value).encode("utf-8")).hexdigest()[:8]
    return f"{prefix}_{digest}"


def is_sensitive_column(column: str) -> bool:
    normalized = column.lower().replace("_", "")
    return any(token in normalized for token in SENSITIVE_TOKENS)


def safe_preview(frame: pd.DataFrame, rows: int = 5, columns: list[str] | None = None) -> pd.DataFrame:
    preview = frame.head(rows).copy() if columns is None else frame[columns].head(rows).copy()
    for column in preview.columns:
        if is_sensitive_column(column):
            preview[column] = preview[column].map(lambda value: stable_mask(value, column))
    return preview

## Preparar notas

A base de notas e normalizada para garantir que `ValorMedia`, `NomePeriodo` e a ordem da etapa estejam em formato adequado. A ordem da etapa e extraida de textos como `1 BIMESTRE`, `2 ETAPA` ou equivalentes.

### Exemplo fixo de como as notas ficam apos a preparacao

Abaixo está uma pequena amostra estática da base de notas já ordenada e com `stage_order` calculado:

| NomePeriodo | NomeCurso | NomeSerie | ApelidoTurma | NomeAluno | NomeDisciplina | NomeEtapa | ValorMedia | stage_order |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 2025 | Ensino Médio | 1ª Série | 1ª Série B - Matutino | MASK_38CC8D | Arte | 1º BIMESTRE | 10.0 | 1 |
| 2025 | Ensino Médio | 1ª Série | 1ª Série B - Matutino | MASK_38CC8D | Arte | 2º BIMESTRE | 8.8 | 2 |
| 2025 | Ensino Médio | 1ª Série | 1ª Série B - Matutino | MASK_38CC8D | Arte | 3º BIMESTRE | 8.1 | 3 |
| 2025 | Ensino Médio | 1ª Série | 1ª Série B - Matutino | MASK_38CC8D | Arte | 4º BIMESTRE | 10.0 | 4 |
| 2025 | Ensino Médio | 1ª Série | 1ª Série B - Matutino | MASK_38CC8D | Biologia | 1º BIMESTRE | 6.8 | 1 |

In [ ]:
grades = prepare_grades(source_tables["grades"])

grade_columns = [
    "NomePeriodo", "NomeSerie", "ApelidoTurma", "IDAluno", "NomeAluno",
    "NomeDisciplina", "NomeEtapa", "stage_order", "ValorMedia",
]
safe_preview(grades, columns=[column for column in grade_columns if column in grades.columns])

### O que aconteceu com a base de notas

Depois de executar esta celula, a base de notas deixa de ser apenas texto bruto e passa a ter ordem temporal interpretavel. `NomePeriodo` vira referencia numerica, `ValorMedia` passa a estar coerente para calculo e `stage_order` cria a sequencia das etapas.

Na pratica, esse e o primeiro passo que permite falar em historico do aluno em vez de linhas soltas de nota.

## Preparar faltas

A base de faltas tambem recebe `stage_order`. Depois, eventos individuais sao agregados para virar atributos de modelagem: faltas da etapa e faltas acumuladas. O agrupamento e feito por aluno, periodo, disciplina e ordem da etapa.

### Exemplo fixo de como as faltas ficam apos a agregacao

Aqui a falta deixa de ser evento isolado e passa a virar contagem por etapa e acumulado:

| IDAluno | NomePeriodo | NomeDisciplina | stage_order | faltas_etapa | faltas_acumuladas |
| --- | --- | --- | --- | --- | --- |
| MASK_9675BF | 2025 | Arte | 2 | 1 | 1 |
| MASK_9675BF | 2025 | Arte | 3 | 2 | 3 |
| MASK_9675BF | 2025 | Biologia | 1 | 3 | 3 |
| MASK_9675BF | 2025 | Biologia | 2 | 4 | 7 |
| MASK_9675BF | 2025 | Biologia | 3 | 7 | 14 |

In [ ]:
absences = prepare_absences(source_tables["absences"])
absence_features = dataset_steps._build_absence_features(absences)

display(pd.DataFrame([
    {"etapa": "faltas_normalizadas", "linhas": len(absences), "colunas": len(absences.columns)},
    {"etapa": "faltas_agregadas", "linhas": len(absence_features), "colunas": len(absence_features.columns)},
]))
safe_preview(absence_features, columns=["IDAluno", "NomePeriodo", "NomeDisciplina", "stage_order", "faltas_etapa", "faltas_acumuladas"])

### O que mudou nas faltas apos a agregacao

Antes, cada linha representava um evento de falta. Depois desta transformacao, a base passa a registrar quantidade de faltas por etapa e acumulado por disciplina.

Essa mudanca e importante porque o modelo nao aprende melhor com milhares de eventos isolados; ele aprende melhor com um sinal consolidado que descreve o comportamento do aluno ate aquele momento.

## Preparar pagamentos

Pagamentos sao normalizados para tipos numericos, booleanos e datas. Depois sao agregados por aluno e ano letivo, evitando que o modelo trabalhe com cada movimento financeiro isolado. Dessa etapa saem indicadores como quantidade de pagamentos, pendencias, proporcao de mensalidades e media de dias ate pagamento.

### Exemplo fixo do resumo financeiro

Aqui o dado financeiro já não é mais transacional: ele vira um resumo anual por aluno:

| IDAluno | NomePeriodo | pagamentos_registrados_ano | pagamentos_pendentes_ano | proporcao_mensalidades_ano | media_valor_pagamento_ano | media_dias_ate_pagamento_ano |
| --- | --- | --- | --- | --- | --- | --- |
| MASK_9675BF | 2025 | 13 | 0 | 0.8461538461538461 | 1736.1538461538462 | 0.0 |
| MASK_9675BF | 2026 | 13 | 9 | 0.8461538461538461 | 1981.5384615384614 | 0.23076923076923078 |
| MASK_7CA835 | 2025 | 14 | 0 | 0.7857142857142857 | 1382.142857142857 | 0.0 |
| MASK_7CA835 | 2026 | 16 | 9 | 0.6875 | 1625.9375 | 0.1875 |
| MASK_F34BBC | 2025 | 14 | 0 | 0.7857142857142857 | 1382.142857142857 | 0.0 |

In [ ]:
payments = prepare_payments(source_tables["payments"])
payment_features = dataset_steps._build_payment_features(payments)

display(pd.DataFrame([
    {"etapa": "pagamentos_normalizados", "linhas": len(payments), "colunas": len(payments.columns)},
    {"etapa": "pagamentos_agregados", "linhas": len(payment_features), "colunas": len(payment_features.columns)},
]))
safe_preview(payment_features)

### O que mudou nos pagamentos apos a preparacao

A base financeira deixa de ser uma lista de movimentos individuais e passa a virar contexto anual por aluno. O resultado esperado aqui e um conjunto mais compacto de indicadores: quantidade de registros, pendencias, proporcao de mensalidades e media de atraso ou antecipacao.

Com isso, o dado financeiro passa a ser utilizavel pela modelagem sem expor cada pagamento individualmente.

## Preparar professores

Os vinculos de professor sao consolidados por unidade, periodo, curso, serie, turma e disciplina. Quando existe mais de um professor, os nomes sao agrupados e a quantidade de professores e registrada. Quando nao existe professor vinculado, o preenchimento final sera feito depois dentro do dataset com um marcador de ausencia legitima.

### Exemplo fixo dos vinculos de professor

A preparacao consolida o vínculo por turma e disciplina para permitir o merge posterior:

| IDSerie | IDTurma | IDDisciplina | NomeFuncionario | quantidade_professores_disciplina |
| --- | --- | --- | --- | --- |
| MASK_B87375 | MASK_6D39E2 | MASK_0C065E | MASK_77D961 | 1 |
| MASK_B87375 | MASK_6D39E2 | MASK_4764F7 | MASK_ED7525 | 1 |
| MASK_B87375 | MASK_6D39E2 | MASK_275E62 | MASK_3A6669 | 1 |
| MASK_B87375 | MASK_6D39E2 | MASK_10344D | MASK_D78877 | 1 |
| MASK_B87375 | MASK_6D39E2 | MASK_FD79BE | MASK_2AD685 | 1 |

In [ ]:
teachers = prepare_teachers(source_tables["teachers"])

display(pd.DataFrame([{"etapa": "professores_consolidados", "linhas": len(teachers), "colunas": len(teachers.columns)}]))
safe_preview(teachers)

### O que aconteceu com os vinculos de professor

Depois desta celula, varios registros detalhados podem se tornar um unico vinculo consolidado por unidade, periodo, curso, serie, turma e disciplina. Quando ha mais de um professor, os nomes sao unidos e a quantidade de docentes fica registrada.

Ou seja, o dado do professor deixa de ser apenas cadastro e vira contexto pedagogico pronto para entrar no dataset final.

## Construir o dataset temporal de modelagem

A funcao `build_prediction_dataset` une tudo e cria as variaveis historicas. O parametro `min_history` define quantas observacoes anteriores o aluno precisa ter naquela disciplina para entrar no dataset final.

Na pipeline operacional, esse valor muda conforme o modo: `previsao_nota` usa 1 por padrao e `alerta_risco` usa 2. Neste notebook, usamos 2 apenas como demonstracao mais conservadora, porque ela evidencia melhor o corte de historico minimo.

### Exemplo fixo do dataset integrado

Depois dos merges, cada linha já combina desempenho, frequência, financeiro e professor:

| IDAluno | NomePeriodo | NomeSerie | NomeDisciplina | NomeEtapa | ValorMedia | faltas_acumuladas | pagamentos_pendentes_ano | NomeFuncionario | target_nota_proxima | target_risco_proxima |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| MASK_9675BF | 2025 | 1ª Série | Arte | 2º BIMESTRE | 8.8 | 1.0 | 0.0 | MASK_ACD990 | 8.1 | 0 |
| MASK_9675BF | 2025 | 1ª Série | Arte | 3º BIMESTRE | 8.1 | 3.0 | 0.0 | MASK_ACD990 | 10.0 | 0 |
| MASK_9675BF | 2025 | 1ª Série | Biologia | 2º BIMESTRE | 7.0 | 7.0 | 0.0 | MASK_1A6CCD | 7.4 | 0 |
| MASK_9675BF | 2025 | 1ª Série | Biologia | 3º BIMESTRE | 7.4 | 14.0 | 0.0 | MASK_1A6CCD | 7.0 | 0 |
| MASK_9675BF | 2025 | 1ª Série | Biologia 1 | 2º BIMESTRE | 7.0 | 7.0 | 0.0 | MASK_8DD3A6 | 5.6 | 1 |

In [ ]:
MODE_DEFAULTS = {
    "previsao_nota": 1,
    "alerta_risco": 2,
}
DEMO_MIN_HISTORY = 2
modeling_dataset = build_prediction_dataset(source_tables, min_history=DEMO_MIN_HISTORY)

pd.DataFrame([{
    "min_history_demonstracao": DEMO_MIN_HISTORY,
    "padrao_previsao_nota": MODE_DEFAULTS["previsao_nota"],
    "padrao_alerta_risco": MODE_DEFAULTS["alerta_risco"],
    "linhas_dataset_modelagem": len(modeling_dataset),
    "colunas_dataset_modelagem": len(modeling_dataset.columns),
    "alunos": modeling_dataset["IDAluno"].nunique() if "IDAluno" in modeling_dataset.columns else None,
    "disciplinas": modeling_dataset["NomeDisciplina"].nunique() if "NomeDisciplina" in modeling_dataset.columns else None,
    "anos_letivos": modeling_dataset["NomePeriodo"].nunique() if "NomePeriodo" in modeling_dataset.columns else None,
}])

### O que nasceu quando o dataset final foi montado

A tabela acima ja nao representa mais um CSV bruto. Ela representa a base de modelagem propriamente dita. A partir daqui, cada linha tem contexto historico suficiente para ensinar algo sobre a proxima etapa do aluno.

Isso significa que varias linhas das bases originais ficaram de fora por dois motivos esperados:
- nao havia proxima nota conhecida para virar alvo;
- o historico minimo da disciplina ainda nao era suficiente.

## Atributos historicos criados

Nesta etapa, cada linha passa a carregar nao apenas a nota atual, mas tambem sinais de historico e contexto. Exemplos: ultima nota, medias historicas, tendencia, faltas acumuladas, comparacao com a coorte, contexto financeiro e informacoes de professor.

In [ ]:
history_columns = [
    "IDAluno", "NomePeriodo", "NomeSerie", "NomeDisciplina", "NomeEtapa",
    "ValorMedia", "nota_anterior_1", "nota_anterior_2", "media_duas_ultimas_notas",
    "media_historica_disciplina", "tendencia_ultima_nota", "faltas_acumuladas",
    "pagamentos_pendentes_ano", "NomeFuncionario", "professor_disponivel",
]

safe_preview(modeling_dataset, columns=[column for column in history_columns if column in modeling_dataset.columns])

### O que os atributos historicos passam a mostrar

A saida acima e a virada mais importante desta fase. O registro atual do aluno agora carrega memoria: notas anteriores, medias historicas, tendencia, faltas acumuladas, contexto financeiro e professor.

Em termos didaticos, o dado deixa de ser uma fotografia isolada e passa a ser uma sequencia resumida em atributos tabulares.

## Alvos de previsao

A preparacao tambem cria os alvos usados na modelagem:

- `target_nota_proxima`: a proxima nota real conhecida na sequencia aluno x disciplina.
- `target_risco_proxima`: 1 quando a proxima nota fica abaixo de 6.0, caso contrario 0.

Isso permite que o projeto tenha dois tipos de resposta: previsao numerica da nota e alerta pedagogico de risco.

### Exemplo fixo dos alvos criados

Aqui a linha atual passa a carregar o que queremos prever na próxima observação:

| IDAluno | NomeDisciplina | ValorMedia | target_nota_proxima | target_risco_proxima | historico_disciplina_count |
| --- | --- | --- | --- | --- | --- |
| MASK_9675BF | Arte | 8.8 | 8.1 | 0 | 1 |
| MASK_9675BF | Arte | 8.1 | 10.0 | 0 | 2 |
| MASK_9675BF | Biologia | 7.0 | 7.4 | 0 | 1 |
| MASK_9675BF | Biologia | 7.4 | 7.0 | 0 | 2 |
| MASK_9675BF | Biologia 1 | 7.0 | 5.6 | 1 | 1 |

In [ ]:
target_columns = [
    "IDAluno", "NomePeriodo", "NomeDisciplina", "NomeEtapa", "ValorMedia",
    "target_nota_proxima", "target_risco_proxima",
    "baseline_ultima_nota", "baseline_media_duas_ultimas", "baseline_hibrido",
]

safe_preview(modeling_dataset, columns=[column for column in target_columns if column in modeling_dataset.columns])

### O que aconteceu quando os alvos foram criados

Depois desta etapa, o projeto passa a ter duas respostas supervisionadas na mesma linha:
- um valor numerico para a proxima nota;
- um indicador binario dizendo se essa proxima nota caiu abaixo de 6.0.

Esse e o momento em que a base deixa de ser apenas descritiva e passa a ser adequada para regressao e classificacao.

## Selecionar colunas para os modelos

Nem toda coluna do dataset entra no treinamento. Identificadores diretos e alvos ficam fora das features. O modelo recebe atributos historicos, contextuais, numericos e categoricos. Na implementacao atual, `NomeSerie`, `NomeDisciplina`, `NomeEtapa` e `NomeFuncionario` entram como categoricas.

### Exemplo fixo das features que seguem para os modelos

Em vez de listar só o começo da lista, a tabela abaixo resume melhor os grupos de atributos usados na modelagem:

| grupo | exemplos |
| --- | --- |
| contexto escolar | NomePeriodo, NomeSerie, NomeDisciplina, NomeEtapa |
| desempenho recente | ValorMedia, nota_anterior_1, nota_anterior_2, nota_anterior_3 |
| tendencia e historico | media_historica_disciplina, tendencia_ultima_nota, desvio_historico_disciplina |
| frequencia | faltas_etapa, faltas_acumuladas |
| professor | NomeFuncionario, quantidade_professores_disciplina, professor_disponivel |
| financeiro | pagamentos_registrados_ano, pagamentos_pendentes_ano, media_valor_pagamento_ano |

In [ ]:
feature_columns, categorical_columns = select_model_columns(modeling_dataset)

pd.DataFrame({
    "coluna": feature_columns,
    "tipo_para_modelo": ["categorica" if column in categorical_columns else "numerica_ou_ordinal" for column in feature_columns],
})

### O que sai para os modelos

A tabela acima mostra que nem tudo que existe no dataset vai para o treinamento. Identificadores diretos e colunas-alvo ficam de fora. O que segue adiante sao apenas as features com potencial de explicar a proxima nota ou o risco.

Essa separacao e importante para evitar vazamento de informacao e para deixar claro o que realmente alimenta a modelagem.

## Comparar volume antes e depois da preparacao

A reducao de linhas e esperada. O dataset final remove linhas sem proxima nota conhecida e linhas que nao atingem o minimo de historico definido. Em outras palavras, a preparacao nao tenta prever qualquer linha bruta; ela preserva apenas observacoes que realmente podem ensinar algo sobre a proxima etapa.

In [ ]:
preparation_volume = pd.DataFrame([
    {"etapa": "notas_originais", "linhas": len(source_tables["grades"])},
    {"etapa": "notas_normalizadas", "linhas": len(grades)},
    {"etapa": "faltas_agregadas", "linhas": len(absence_features)},
    {"etapa": "pagamentos_agregados", "linhas": len(payment_features)},
    {"etapa": "professores_consolidados", "linhas": len(teachers)},
    {"etapa": "dataset_modelagem", "linhas": len(modeling_dataset)},
])

preparation_volume

### Como ler a reducao de volume

Se o numero de linhas caiu, isso nao significa perda indevida. Significa filtragem metodologica. O projeto esta descartando observacoes que nao servem para prever a proxima etapa com criterio temporal.

Na leitura didatica da banca, este quadro mostra exatamente onde o dado foi ficando mais analitico e menos bruto.

## Opcional: salvar uma amostra demonstrativa mascarada

Por padrao, este notebook nao salva dados. Se for necessario criar um exemplo para apresentacao, ative `SAVE_MASKED_SAMPLE = True`. Mesmo assim, revise o arquivo antes de publicar.

In [ ]:
SAVE_MASKED_SAMPLE = False

if SAVE_MASKED_SAMPLE:
    sample_columns = [column for column in target_columns if column in modeling_dataset.columns]
    masked_sample = safe_preview(modeling_dataset, rows=20, columns=sample_columns)
    output_path = PROJECT_ROOT / "artifacts" / "crisp_dm" / "data_preparation_masked_sample.csv"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    masked_sample.to_csv(output_path, index=False, encoding="utf-8")
    print(output_path)

## Resultado da fase

Ao final da preparacao, o projeto deixa de ter varias tabelas separadas e passa a ter uma base temporal unica, pronta para a fase de Modeling. Essa base contem:

- nota atual e historico anterior;
- faltas da etapa e acumuladas;
- informacoes financeiras agregadas;
- professor vinculado ou marcador de ausencia de vinculo;
- comparacao com media da coorte;
- alvo numerico de proxima nota;
- alvo binario de risco academico;
- baselines simples para comparacao honesta com os modelos.

Com isso, a fase seguinte pode comparar algoritmos diferentes sem precisar rediscutir a integracao das bases, porque a estrutura temporal de entrada ja esta definida.